# Fetching a data from table

In [ ]:
# imports
import streamlit as st
import os, tempfile
import pandas as pd
from langchain.chat_models import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import CSVLoader
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain.chains.summarize import load_summarize_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_experimental.agents import create_pandas_dataframe_agent
import asyncio

st.set_page_config(page_title="CSV AI", layout="wide")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def home_page():
    st.write("""Select any one feature from above sliderbox: \n
    1. Chat with CSV \n
    2. Summarize CSV \n
    3. Analyze CSV  """)

@st.cache_resource()
def retriever_func(uploaded_file):
    if uploaded_file :
        with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
            tmp_file.write(uploaded_file.getvalue())
            tmp_file_path = tmp_file.name
        try:
            loader = CSVLoader(file_path=tmp_file_path, encoding="utf-8")
            data = loader.load()
        except:
            loader = CSVLoader(file_path=tmp_file_path, encoding="cp1252")
            data = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(
                        chunk_size=1000, 
                        chunk_overlap=200, 
                        add_start_index=True
                        )
        all_splits = text_splitter.split_documents(data)

        
        vectorstore = FAISS.from_documents(documents=all_splits, embedding=OpenAIEmbeddings())
        retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})
    if not uploaded_file:
        st.info("Please upload CSV documents to continue.")
        st.stop()
    return retriever, vectorstore

def chat(temperature, model_name):
    st.write("# Talk to CSV")
    # Add functionality for Page 1
    reset = st.sidebar.button("Reset Chat")
    uploaded_file = st.sidebar.file_uploader("Upload your CSV here 👇:", type="csv")
    retriever, vectorstore = retriever_func(uploaded_file)
    llm = ChatOpenAI(model_name=model_name, temperature=temperature, streaming=True)
        
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    if "messages" not in st.session_state:
        st.session_state["messages"] = [{"role": "assistant", "content": "How can I help you?"}]

    store = {}

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """Use the following pieces of context to answer the question at the end.
                  If you don't know the answer, just say that you don't know, don't try to make up an answer. Context: {context}""",
            ),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{input}"),
        ]
    )
    runnable = prompt | llm
    
    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]


    with_message_history = RunnableWithMessageHistory(
        runnable,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    for msg in st.session_state.messages:
        st.chat_message(msg["role"]).write(msg["content"])
    async def chat_message():
        if prompt := st.chat_input():
            if not user_api_key: 
                st.info("Please add your OpenAI API key to continue.")
                st.stop()
            st.session_state.messages.append({"role": "user", "content": prompt})
            st.chat_message("user").write(prompt)
            contextt = vectorstore.similarity_search(prompt, k=6)
            context = "\n\n".join(doc.page_content for doc in contextt)
            #msg = 
            with st.chat_message("assistant"):
                message_placeholder = st.empty()
                text_chunk = ""
                async for chunk in with_message_history.astream(
                        {"context": context, "input": prompt},
                        config={"configurable": {"session_id": "abc123"}},
                    ):
                    text_chunk += chunk.content
                    message_placeholder.markdown(text_chunk)
                    #st.chat_message("assistant").write(text_chunk)
                st.session_state.messages.append({"role": "assistant", "content": text_chunk})
        if reset:
            st.session_state["messages"] = []
    asyncio.run(chat_message())


def summary(model_name, temperature, top_p):
    st.write("# Summary of CSV")
    st.write("Upload your document here:")
    uploaded_file = st.file_uploader("Upload source document", type="csv", label_visibility="collapsed")
    if uploaded_file is not None:
        with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
            tmp_file.write(uploaded_file.getvalue())
            tmp_file_path = tmp_file.name
        # encoding = cp1252
        text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024, chunk_overlap=100)
        try:
            loader = CSVLoader(file_path=tmp_file_path, encoding="cp1252")
            #loader = UnstructuredFileLoader(tmp_file_path)
            data = loader.load()
            texts = text_splitter.split_documents(data)
        except:
            loader = CSVLoader(file_path=tmp_file_path, encoding="utf-8")
            #loader = UnstructuredFileLoader(tmp_file_path)
            data = loader.load()
            texts = text_splitter.split_documents(data)

        os.remove(tmp_file_path)
        gen_sum = st.button("Generate Summary")
        if gen_sum:
            # Initialize the OpenAI module, load and run the summarize chain
            llm = ChatOpenAI(model_name=model_name, temperature=temperature)
            chain = load_summarize_chain(
                llm=llm,
                chain_type="map_reduce",

                return_intermediate_steps=True,
                input_key="input_documents",
                output_key="output_text",
            )
            result = chain({"input_documents": texts}, return_only_outputs=True)

            st.success(result["output_text"])


def analyze(temperature, model_name):
    st.write("# Analyze CSV")
    #st.write("This is Page 3")
    # Add functionality for Page 3
    reset = st.sidebar.button("Reset Chat")
    uploaded_file = st.sidebar.file_uploader("Upload your CSV here 👇:", type="csv")
    #.write(uploaded_file.name)
    if uploaded_file is not None:
        with tempfile.NamedTemporaryFile(delete=False) as tmp_file:
            tmp_file.write(uploaded_file.getvalue())
            tmp_file_path = tmp_file.name
        df = pd.read_csv(tmp_file_path)
        llm = ChatOpenAI(model=model_name, temperature=temperature)
        agent = create_pandas_dataframe_agent(llm, df, agent_type="openai-tools", verbose=True)

        if "messages" not in st.session_state:
            st.session_state["messages"] = [{"role": "assistant", "content": "How can I help you?"}]
            
        for msg in st.session_state.messages:
            st.chat_message(msg["role"]).write(msg["content"])

        if prompt := st.chat_input(placeholder="What are the names of the columns?"):
            if not user_api_key: 
                st.info("Please add your OpenAI API key to continue.")
                st.stop()
            st.session_state.messages.append({"role": "user", "content": prompt})
            st.chat_message("user").write(prompt)
            msg = agent.invoke({"input": prompt, "chat_history": st.session_state.messages})
            st.session_state.messages.append({"role": "assistant", "content": msg["output"]})
            st.chat_message("assistant").write(msg["output"])
        if reset:
            st.session_state["messages"] = []


# Main App
def main():
    st.markdown(
        """
        <div style='text-align: center;'>
            <h1>🧠 CSV AI</h1>
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.markdown(
        """
        <div style='text-align: center;'>
            <h4>⚡️ Interacting, Analyzing and Summarizing CSV Files!</h4>
        </div>
        """,
        unsafe_allow_html=True,
    )
    global user_api_key
    # #
    # st.sidebar.write("---Made with ❤️---")
    # st.sidebar.write("---")
    if os.path.exists(".env") and os.environ.get("OPENAI_API_KEY") is not None:
        user_api_key = os.environ["OPENAI_API_KEY"]
        st.success("API key loaded from .env", icon="🚀")
    else:
        user_api_key = st.sidebar.text_input(
            label="#### Enter OpenAI API key 👇", placeholder="Paste your openAI API key, sk-", type="password", key="openai_api_key"
        )
        if user_api_key:
            st.sidebar.success("API key loaded", icon="🚀")

    os.environ["OPENAI_API_KEY"] = user_api_key

    

    # Execute the home page function
    MODEL_OPTIONS = ["gpt-3.5-turbo", "gpt-4", "gpt-4-32k","gpt-3.5-turbo-16k","gpt-4-1106-preview"]
    max_tokens = {"gpt-4":7000, "gpt-4-32k":31000, "gpt-3.5-turbo":3000}
    TEMPERATURE_MIN_VALUE = 0.0
    TEMPERATURE_MAX_VALUE = 1.0
    TEMPERATURE_DEFAULT_VALUE = 0.9
    TEMPERATURE_STEP = 0.01
    model_name = st.sidebar.selectbox(label="Model", options=MODEL_OPTIONS)
    top_p = st.sidebar.slider("Top_P", 0.0, 1.0, 1.0, 0.1)
    # freq_penalty = st.sidebar.slider("Frequency Penalty", 0.0, 2.0, 0.0, 0.1)
    temperature = st.sidebar.slider(
                label="Temperature",
                min_value=TEMPERATURE_MIN_VALUE,
                max_value=TEMPERATURE_MAX_VALUE,
                value=TEMPERATURE_DEFAULT_VALUE,
                step=TEMPERATURE_STEP,)

    # Define a dictionary with the function names and their respective functions
    functions = [
        "home",
        "Chat with CSV",
        "Summarize CSV",
        "Analyze CSV",
    ]
    
    #st.subheader("Select any generator👇")
    # Create a selectbox with the function names as options
    selected_function = st.selectbox("Select a functionality", functions)
    if selected_function == "home":
        home_page()
    elif selected_function == "Chat with CSV":
        chat(temperature=temperature, model_name=model_name)
    elif selected_function == "Summarize CSV":
        summary(model_name=model_name, temperature=temperature, top_p=top_p)
    elif selected_function == "Analyze CSV":
        analyze(temperature=temperature, model_name=model_name)
    else:
        st.warning("You haven't selected any AI Functionality!!")
    

    

if __name__ == "__main__":
    main()

In [42]:
import pymssql
import pandas as pd

server = "172.16.0.66" 
database = "ds_test"
username = "datascience"
password = "mtbc@1234"

try:
    conn = pymssql.connect(server=server, user=username, password=password, database=database)
    cursor = conn.cursor()
    print("Connection successful!")
    cursor.execute("SELECT TOP 5 * FROM users")
    rows = cursor.fetchall()
    columns = [col[0] for col in cursor.description]
    df = pd.DataFrame(rows, columns=columns)

except pymssql.DatabaseError as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"Error: {e}")
finally:
    if conn:
        conn.close()
        print("Connection closed.")


Connection successful!
Connection closed.


In [44]:
df

,index,User_Id,Password,Employee_Id,User_FName,User_LName,User_MI,Email,Team_Id,Company_Id,...,LAST_LOGIN_DATE,LOCKEDBY,IS_LOCKED_OUT,FAILED_PASSWORD_ATTEMPT_COUNT,COMMENTS,NAME_ON_LETTER,PHONE_ON_LETTER,EXT_ON_LETTER,Is_Websoft,Platform
0,0,AA142,5A9A6CECF7DCDB3F371E6D221A3DEF0525FE8945362B65...,5366,ASIF,ALI,,ASIFALI4@CARECLOUD.COM,5481022.0,10,...,2023-12-18 14:05:08.680,,False,0.0,Login Successfull.,None,None,None,None,
1,1,AA153,25B26B1EC6263AFD720BA467A71271EB10302135E4AC50...,5636,ALI,AZHAR,,ALIAZHAR@CARECLOUD.COM,534412.0,10,...,2023-12-04 14:00:19.983,,False,0.0,Logout Successfull.,None,None,None,None,None
2,2,AA163,CF7BBB0276657A41DF25C8612E437A6FD8C89D6F39A44A...,5749,AMMAR,ALI,,AMMARALI@CARECLOUD.COM,5481535.0,10,...,2024-01-31 15:20:26.680,,False,0.0,Logout Successfull.,None,None,None,None,
3,3,AA167,1393D640DAEBF7DD67AD41AF732DB954EC0F673C66EAE6...,5796,AFZAL,AHMED,,AFZALAHMED@CARECLOUD.COM,548793.0,17,...,2023-12-18 13:59:10.063,,False,0.0,Login Successfull.,None,None,None,None,OTHER
4,4,AA199,42B9B70467CD59FDD47947CFC179A0EBCF4793BD95990C...,6425,ANJUM,AZAD,,ANJUMAZAD@CARECLOUD.COM,548279.0,10,...,2023-12-18 07:10:33.797,,False,0.0,Login Successfull.,None,None,None,None,


# Creating a Table

In [48]:
import pymssql
import pandas as pd

server = "172.16.0.66"
database = "ds_test"
username = "datascience"
password = "mtbc@1234"

try:
    conn = pymssql.connect(server=server, user=username, password=password,  database=database)
    cursor = conn.cursor()
    print("Connection successful!")
    create_table_query = """
    CREATE TABLE test_tb_ibrahim (
        id INT IDENTITY(1,1) PRIMARY KEY,`
        name VARCHAR(50),
        email VARCHAR(100) UNIQUE,
        age INT
    )
    """
    cursor.execute(create_table_query)
    conn.commit()

    print("Table 'test_tb_ibrahim' created successfully!")
    cursor.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'test_tb_ibrahim'")
    table_exists = cursor.fetchone()

    if table_exists:
        print("Table exists in the database!")
    else:
        print("Table creation failed.")

except pymssql.DatabaseError as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"Error: {e}")
finally:
    if conn:
        conn.close()
        print("Connection closed.")


Connection successful!
Table 'test_tb_ibrahim' created successfully!
Table exists in the database!
Connection closed.


# Deleting a table 

In [47]:
import pymssql


server = "172.16.0.66"
database = "ds_test"
username = "datascience"
password = "mtbc@1234"

try:
    conn = pymssql.connect(server=server, user=username, password=password, database=database)
    cursor = conn.cursor()

    print("Connection successful!")
    cursor.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'test_tb_ibrahim'")
    table_exists = cursor.fetchone()

    if table_exists:
        drop_table_query = "DROP TABLE test_tb_ibrahim"
        cursor.execute(drop_table_query)
        conn.commit()
        print("Table 'test_tb_ibrahim' deleted successfully!")
        cursor.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'test_tb_ibrahim'")
        table_check = cursor.fetchone()

        if table_check is None:
            print("Table deletion confirmed.")
        else:
            print("Table still exists!")

    else:
        print("Table does not exist.")

except pymssql.DatabaseError as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"Error: {e}")
finally:
    if conn:
        conn.close()
        print("Connection closed.")


Connection successful!
Table 'test_users' deleted successfully!
Table deletion confirmed.
Connection closed.
